In [5]:
import zipfile
import shutil
from pathlib import Path
import pandas as pd

zip_path = "./data/zip/"
save_path = "./data/split/"
datasets = ['ml-100k', 'jester', 'amazon_all_beauty', 'netflix']
datatypes = ['explicit', 'implicit']
datatype = datatypes[1]
dataset = datasets[2]

def unzip(zip_path: str):
    """Unzip a specific file, print 'Unzipped', then delete the extracted files."""
    zip_path = Path(zip_path)
    if not zip_path.exists() or zip_path.suffix != '.zip':
        print(f"❌ File not found or not a .zip: {zip_path}")
        return

    # Extraction folder (same name as zip)
    extract_dir = zip_path.parent / zip_path.stem

    # Unzip the file
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("✅ extract_dir Unzipped")

def cleanup(folder_path: str):
    folder_path = Path(folder_path)
    if folder_path.exists():
            shutil.rmtree(folder_path)
            print(f"🧹 Deleted extracted folder: {folder_path}")

unzip(f"{zip_path}{dataset}.zip")

# --- Load the .inter file ---
df_inter = pd.read_csv(f"{zip_path}{dataset}/{dataset}.inter", sep="\t")

# --- Keep only desired collumns ---
desired_cols = ['user_id:token', 'item_id:token', 'rating:float']
df_inter = df_inter[[c for c in desired_cols if c in df_inter.columns]]
print("✅ .inter file opened")

# --- Load the .item file ---
df_item = None
if dataset=='jester':
    df_item = pd.DataFrame({"item_id:token": [i for i in range(0,101)]})
elif dataset=='netflix':
    unique_items = df_inter['item_id:token'].unique()
    df_item = pd.DataFrame({'item_id:token': unique_items})
else:
    df_item = pd.read_csv(f"{zip_path}{dataset}/{dataset}.item", sep="\t")
    df_item = df_inter[['item_id:token']]
print("✅ .item file opened")

# --- Filtering users ---
users_before = df_inter['user_id:token'].nunique()
rows_before = len(df_inter)

if datatype == datatypes[1]:   # implicit 
    # Hard cutoff
    # df_inter = df_inter[df_inter['rating:float'] >= 2.0].copy()

    # Pr. user mean cutoff
    user_mean = df_inter.groupby('user_id:token')['rating:float'].transform('mean')
    df_inter = df_inter[df_inter['rating:float'] >= user_mean].copy()

    df_inter.loc[:, 'rating:float'] = 1

# explicit and implicit
user_item_counts = df_inter.groupby('user_id:token')['item_id:token'].nunique()
unique_items_count = df_item['item_id:token'].nunique()

users_too_few = user_item_counts[user_item_counts < 5].index
users_all_items = user_item_counts[user_item_counts >= unique_items_count].index

bad_users = set(users_too_few).union(set(users_all_items))

df_inter = df_inter[~df_inter['user_id:token'].isin(bad_users)]

users_after = df_inter['user_id:token'].nunique()
rows_after = len(df_inter)
print("✅ users filtered")

# --- Assign split ---
def assign_splits(df_inter: pd.DataFrame, seed: int = 22) -> pd.DataFrame:
    df = df_inter.copy()
    df['split:token'] = 'train'

    # per-user interaction counts
    cnt = df.groupby('user_id:token')['item_id:token'].nunique()

    # users eligible for val/test
    users_for_val = cnt.index[cnt >= 1]
    users_for_test = cnt.index[cnt >= 2]

    # pick 1 val row per eligible user
    val_idx = (df[df['user_id:token'].isin(users_for_val)].groupby('user_id:token', group_keys=False).sample(n=1, random_state=seed).index)
    df.loc[val_idx, 'split:token'] = 'val'

    # from the remaining rows, pick 1 test row per eligible user
    remaining = df.drop(index=val_idx)
    test_idx = (remaining[remaining['user_id:token'].isin(users_for_test)].groupby('user_id:token', group_keys=False).sample(n=1, random_state=seed + 1).index)
    df.loc[test_idx, 'split:token'] = 'test'

    return df
df_inter = assign_splits(df_inter)
print("✅ splits assigned")

# --- save files ---
explicit_path = f"{save_path}{datatype}/{dataset}/{dataset}"
Path(explicit_path).parent.mkdir(parents=True, exist_ok=True)
df_inter.to_csv(f"{explicit_path}.inter", sep='\t', index=False)
df_item.to_csv(f"{explicit_path}.item", sep='\t', index=False)
print("✅ explicit files saved")

cleanup(f"{zip_path}{dataset}")

print(f"users removed {users_before - users_after}")
print(f"rows removed {rows_before - rows_after}")

✅ extract_dir Unzipped
✅ .inter file opened
✅ .item file opened
✅ users filtered
✅ splits assigned
✅ explicit files saved
🧹 Deleted extracted folder: data\zip\amazon_all_beauty
users removed 631207
rows removed 686769
